In [307]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [308]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

## 1. Load dataset

In [309]:
df = pd.read_csv('data/student.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [310]:
X = df.drop(columns=['math_score'])
y = df['math_score']

## 2. Train test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)

In [312]:
X_train.columns

Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course', 'reading_score', 'writing_score'],
      dtype='str')

## 3. Feature groups

In [313]:
numeric_features = X_train.select_dtypes(exclude=['object', 'string']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"Numeric features: {numeric_features}")
print(f"categorical features: {categorical_features}")

Numeric features: ['reading_score', 'writing_score']
categorical features: ['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course']


In [314]:
X_train['parental_level_of_education'].unique()

<StringArray>
[      'some college',        'high school',  'bachelor's degree',
 'associate's degree',   'some high school',    'master's degree']
Length: 6, dtype: str

In [315]:
ordinal_features = ['parental_level_of_education']
education_order = ['some high school', 'high school', 'some college', "associate's degree", "bachelor's degree", "master's degree"]
nominal_features = [col for col in categorical_features if col not in ordinal_features]

In [316]:
print(f"Nominal features: {nominal_features}")

Nominal features: ['gender', 'race_ethnicity', 'lunch', 'test_preparation_course']


## 4. Pipelines per feature type

In [317]:
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())])
nominal_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown="ignore", drop='first', sparse_output=True))])
ordinal_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(categories=[education_order], handle_unknown="use_encoded_value", unknown_value=-1))])

In [318]:
preprocessing = ColumnTransformer([('numeric', numeric_pipeline, numeric_features), ('nominal', nominal_pipeline, nominal_features), ('ordinal', ordinal_pipeline, ordinal_features)])

## 5. Full pipeline with model

In [319]:
model = LinearRegression()

model_pipeline = Pipeline([('preprocessing', preprocessing), ('model', model)])

## 6. Fit the model

In [320]:
model_pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('nominal', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 7. Model evaluation on test set

In [321]:
def evaluate_model(X, y):
    y_pred = model_pipeline.predict(X)
    mse = mean_squared_error(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    return mse, mae, r2

## 8. Testing various models

In [322]:
models = {'Linear Regression': LinearRegression(),
'Lasso': Lasso(),
'Ridge': Ridge(),
'Decision Tree': DecisionTreeRegressor(),
'Random Forest': RandomForestRegressor(),
'Ada Boost': AdaBoostRegressor(),
'K Neighbors': KNeighborsRegressor(),
'SVR': SVR(),
'Catboost': CatBoostRegressor(),
'Xgboost': XGBRegressor()}

In [323]:
eval_metrics = pd.DataFrame(np.nan, index=models.keys(), columns=['train_mse', 'train_mae', 'train_r2', 'test_mse', 'test_mae', 'test_r2'])

In [ ]:
for k, v in models.items():
    model_pipeline = Pipeline([('preprocessing', preprocessing), ('model', v)])
    model_pipeline.fit(X_train, y_train)
    print(f"Training Model: {k}\n")
    mse, mae, r2 = evaluate_model(X_train, y_train)
    eval_metrics.loc[k, 'train_mse'] = mse
    eval_metrics.loc[k, 'train_mae'] = mae
    eval_metrics.loc[k, 'train_r2'] = r2

    mse, mae, r2 = evaluate_model(X_test, y_test)
    eval_metrics.loc[k, 'test_mse'] = mse
    eval_metrics.loc[k, 'test_mae'] = mae
    eval_metrics.loc[k, 'test_r2'] = r2

--------------------------------------
Training Model: Linear Regression

Training Model: Lasso

Training Model: Ridge

Training Model: Decision Tree

Training Model: Random Forest

Training Model: Ada Boost

Training Model: K Neighbors

Training Model: SVR

Learning rate set to 0.039525
0:	learn: 14.8954029	total: 693us	remaining: 693ms
1:	learn: 14.4902512	total: 1.28ms	remaining: 637ms
2:	learn: 14.1379395	total: 1.85ms	remaining: 615ms
3:	learn: 13.7786087	total: 2.42ms	remaining: 603ms
4:	learn: 13.4038297	total: 2.96ms	remaining: 590ms
5:	learn: 13.0629579	total: 3.51ms	remaining: 582ms
6:	learn: 12.7899288	total: 4.11ms	remaining: 583ms
7:	learn: 12.4866870	total: 4.67ms	remaining: 579ms
8:	learn: 12.1688985	total: 5.23ms	remaining: 576ms
9:	learn: 11.8586294	total: 5.77ms	remaining: 571ms
10:	learn: 11.5823901	total: 6.33ms	remaining: 569ms
11:	learn: 11.3122489	total: 6.89ms	remaining: 567ms
12:	learn: 11.1062482	total: 7.46ms	remaining: 566ms
13:	learn: 10.8916023	total: 8ms	

In [332]:
eval_metrics.sort_values(by='test_r2', ascending=False)

,train_mse,train_mae,train_r2,test_mse,test_mae,test_r2
Ridge,27.683850,4.191183,0.881906,31.696767,4.520806,0.849252
Linear Regression,27.679147,4.190166,0.881926,31.714095,4.525041,0.849170
Catboost,9.188661,2.348648,0.960803,37.116427,4.920299,0.823477
Random Forest,5.062578,1.778006,0.978404,39.190432,5.000033,0.813613
Ada Boost,32.440794,4.614707,0.861614,41.849242,5.151447,0.800968
SVR,46.936731,5.142434,0.799778,43.344110,5.231700,0.793858
Lasso,41.854900,5.108302,0.821456,43.689100,5.213810,0.792218
Xgboost,0.844167,0.585941,0.996399,49.029346,5.550318,0.766820
K Neighbors,32.770850,4.539250,0.860206,51.301800,5.793000,0.756012
Decision Tree,0.185000,0.030000,0.999211,67.900000,6.540000,0.677072
